In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import h5py
import os
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Load metadata
meta = pd.read_csv("/content/drive/Shareddrives/EMG POSE deep learning/emg2pose_dataset_mini/metadata.csv")

In [ ]:
pd.set_option('display.max_colwidth', None)
meta['filename'].sample(10)

,filename
23061,2022-11-21-1669017600-3a415-cv-emg-pose-train@2-recording-13_left
21812,2022-11-11-1668153600-a4656-cv-emg-pose-train@2-recording-15_right
141,2022-04-12-1649750400-19a77-cv-emg-pose-demonstration@2-recording-1_right
9144,2022-08-03-1659513600-4ac73-cv-emg-pose-demonstration@2-recording-18_right
13083,2022-09-15-1663228800-a3e92-cv-emg-pose-train@2-recording-7_left
2595,2022-06-13-1655107200-9c839-cv-emg-pose-train@2-recording-14_right
24693,2022-12-02-1669968000-d1d88-cv-emg-pose-train@2-recording-10_left
870,2022-05-17-1652774400-f319c-cv-emg-pose-demonstration@2-recording-18_left
14423,2022-09-22-1663833600-c51b4-cv-emg-pose-demonstration@2-recording-1_left
797,2022-05-17-1652774400-2e354-cv-emg-pose-demonstration@2-recording-16_right


In [ ]:
meta['type'] = meta['filename'].str.extract(r'pose-(.*?)@')
meta['type'].value_counts() # what are demonstration and train?

,count
type,
demonstration,13772
train,11481


In [ ]:
print(pd.crosstab(
    index=[meta['held_out_user'], meta['held_out_stage']],
    columns=meta['split']
))

print(pd.crosstab(
    index=[meta['held_out_user'], meta['held_out_stage']],
    columns=meta['generalization']
))

print(pd.crosstab(meta['generalization'], meta['split']
))

split                         test  train   val
held_out_user held_out_stage                   
False         False              0  17136     0
              True            3539      0     0
True          False           2172      0  1618
              True             456      0   332
generalization                 none  stage  user  user_stage
held_out_user held_out_stage                                
False         False           17136      0     0           0
              True                0   3539     0           0
True          False               0      0  3790           0
              True                0      0     0         788
split           test  train   val
generalization                   
none               0  17136     0
stage           3539      0     0
user            2172      0  1618
user_stage       456      0   332


In [ ]:
print(pd.crosstab(meta['generalization'], meta['split']))


split           test  train   val
generalization                   
none               0  17136     0
stage           3539      0     0
user            2172      0  1618
user_stage       456      0   332


**generalization**

`none`: no held_out_user & no held_out_stage, used for train.  
`stage`: no held_out_user & held_out_stage, used for test.  
`user`: held_out_user & no hled_out_stage, used for validation and test.  
`user_stage`: held_out_user & held_out_stage, used for validation and test.





In [ ]:
# 1. Generate the table with unique user counts
unique_user_table = pd.crosstab(
    index=meta['generalization'],
    columns=meta['split'],
    values=meta['user'],
    aggfunc='nunique'
).fillna(0).astype(int)

# 2. Reorder the columns to 'train', 'val', 'test'
ordered_table = unique_user_table.reindex(columns=['train', 'val', 'test'])

print(ordered_table)

split           train  val  test
generalization                  
none              158    0     0
stage               0    0   158
user                0   15    20
user_stage          0   15    20


User Distribution
* Non-Held-Out Users ($n=158$):
  * Contributed data to both non-held-out and held-out stages.
  * Held-out stage data used for stage-level test only (generalization = `stage` & split = `test`)
  * No stage-level validation.
* Held-Out Users ($n=35$):
  * Contributed data to both stages but were entirely excluded from the training set.
  * Validation ($n=15$): Allocated for both user- and user-stage-level validation ((generalization = `user` or `user_stage`) & split = `val`).
  * Testing ($n=20$): Allocated for both user- and user-stage-level test ((generalization = `user` or `user_stage`) & split = `test`).

In [ ]:
# 1. Generate the table with unique stage counts
unique_user_table = pd.crosstab(
    index=meta['generalization'],
    columns=meta['split'],
    values=meta['stage'],
    aggfunc='nunique'
).fillna(0).astype(int)

# 2. Reorder the columns to 'train', 'val', 'test'
ordered_table = unique_user_table.reindex(columns=['train', 'val', 'test'])

print(ordered_table)

split           train  val  test
generalization                  
none               23    0     0
stage               0    0     6
user                0   23    23
user_stage          0    6     6


There are 23 non-held-out stages and 6 held-out-stages.

In [ ]:
print(pd.crosstab(meta['generalization'], meta['type']))
print(pd.crosstab(meta['split'], meta['type']))

type            demonstration  train
generalization                      
none                    10236   6900
stage                    1032   2507
user                     2272   1518
user_stage                232    556
type   demonstration  train
split                      
test            2480   3687
train          10236   6900
val             1056    894


`demonstration` and `train` in the `filename` do not seem to be related to generalization or split.

In [ ]:
check_cols = ['held_out_user', 'held_out_stage', 'split', 'generalization', 'type']

# Group by user and count unique combinations of these columns
user_variations = meta.groupby('user')[check_cols].nunique()

user_variations.head(10)


,held_out_user,held_out_stage,split,generalization,type
user,,,,,
01aff754c1,1,2,2,2,2
0219627c14,1,2,1,2,2
03c48f96aa,1,2,1,2,2
03d49b393e,1,2,2,2,2
05026be68a,1,2,1,2,2
055bb1b487,1,2,2,2,2
057e6195fe,1,2,2,2,2
059f336bce,1,2,1,2,2
0625995630,1,2,2,2,2


In [ ]:
# Iterate through each column and print the frequency of the counts
for col in user_variations.columns:
    print(f"--- Frequency Table for: {col} ---")
    print(user_variations[col].value_counts().sort_index())
    print("\n")

--- Frequency Table for: held_out_user ---
held_out_user
1    193
Name: count, dtype: int64


--- Frequency Table for: held_out_stage ---
held_out_stage
2    193
Name: count, dtype: int64


--- Frequency Table for: split ---
split
1     35
2    158
Name: count, dtype: int64


--- Frequency Table for: generalization ---
generalization
2    193
Name: count, dtype: int64


--- Frequency Table for: type ---
type
1      4
2    189
Name: count, dtype: int64




In [ ]:
valid_users = user_variations[user_variations['split'] == 1].index
meta[meta['user'].isin(valid_users)]['split'].value_counts()

,count
split,
test,2628
val,1950


- Every user has both held-out stages and non-held-out stages.
- 158 users have train and test data.